In [1]:
import torch
import torch.nn 
import torch.nn.functional
from typing import Union, List

batch_size = 64
context_len = 256

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
device

'cuda'

# Architettura e Implementazione di un Transformer (Decoder-only)

## 1. Rappresentazione del Dato e Pre-processing
L'obiettivo primario è la mappatura di un corpus testuale in uno spazio vettoriale computabile dal modello.

* **Tokenizzazione a livello di Carattere**: Identificazione del set di caratteri univoci (vocabolario) per definire la dimensionalità dello spazio di input.
* **Encoding Bidirezionale**: Creazione di funzioni di mapping (`token_to_id` e `id_to_token`) per tradurre i token in indici interi e viceversa.
* **Configurazione degli Iperparametri**: Definizione della dimensione dei batch (parallelismo) e della lunghezza del contesto (finestra di attenzione), parametri che condizionano la memoria e la capacità computazionale.
* **Suddivisione del Dataset**: Partizionamento tra training e validation set per garantire una corretta valutazione della capacità di generalizzazione.

PS: Nella `token_to_it` usa `"".join`

In [3]:
with open("../data/shakespeare.txt", "r") as f:
    raw_data = f.read()

In [4]:
tokens = sorted(set(raw_data))
vocab_size = len(tokens)

In [5]:
token_to_id = { el: i for i, el in enumerate(tokens)}
id_to_token = { i: el for i, el in enumerate(tokens)}

In [6]:
encode = lambda x: torch.tensor([ token_to_id[s] for s in x])
decode = lambda x: "".join([ id_to_token[s] for s in x])

In [7]:
encoded_data = encode(raw_data)

len_data = len(encoded_data)

train_data = encoded_data[:int(0.8 * len_data)]
test_data  = encoded_data[int(0.8 * len_data):]

In [8]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    #select a starting point
    ix = torch.randint(len(data) - context_len, (batch_size,))

    #select starting point + context_len
    x = torch.stack([data[i:i+context_len] for i in ix])

    #select target
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
get_batch("train")

(tensor([[58, 43, 50,  ..., 58,  1, 39],
         [59, 56, 42,  ...,  1, 39, 58],
         [ 1, 46, 39,  ..., 59, 56,  1],
         ...,
         [44,  1, 39,  ..., 58, 46, 11],
         [ 1, 53, 44,  ..., 10,  0,  5],
         [52, 42, 43,  ..., 42,  1, 58]], device='cuda:0'),
 tensor([[43, 50, 63,  ...,  1, 39, 52],
         [56, 42, 43,  ..., 39, 58,  1],
         [46, 39, 58,  ..., 56,  1, 43],
         ...,
         [ 1, 39,  1,  ..., 46, 11,  0],
         [53, 44,  1,  ...,  0,  5, 32],
         [42, 43, 56,  ...,  1, 58, 46]], device='cuda:0'))

## 3. Scaled Dot-Product Attention
Il cuore del Transformer risiede nella gestione della focalizzazione del modello sulle parti rilevanti della sequenza tramite la classe `Head`.

$$
 \text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}} \right)V
$$

* **Proiezioni Lineari (Q, K, V)**: Trasformazione dell'input in vettori *Query*, *Key* e *Value* tramite matrici di peso dedicate.
* **Calcolo della Matrice di Affinità**: Prodotto scalare tra $Q$ e $K$, normalizzato per la radice quadrata della dimensione della testa per mantenere la stabilità dei gradienti.
* **Causal Masking**: Applicazione di una maschera triangolare inferiore per forzare l'attenzione a ignorare i token futuri, rispettando il vincolo di causalità del decoder.
* **Distribuzione di Probabilità**: Utilizzo della funzione Softmax per pesare l'influenza dei diversi token sulla rappresentazione finale.

## 4. Modularità e Composizione: Il Transformer Block
Integrazione di componenti modulari per facilitare l'apprendimento di feature complesse.

* **Multi-Head Attention**: Implementazione di più teste di attenzione in parallelo (es. 6 teste) per catturare simultaneamente diverse relazioni semantiche.
* **Feed-Forward Network (FFN)**: Stadio di elaborazione locale che applica trasformazioni lineari e non lineari (ReLU) ad ogni posizione della sequenza indipendentemente.
* **Stabilità Numerica**: Integrazione di *Layer Normalization* e connessioni residue (*Residual Connections*) per mitigare il problema della scomparsa del gradiente in architetture profonde.

## 5. Architettura Globale e Generazione
Integrazione finale dei moduli per la creazione del modello completo.

* **Embedding Combinato**: Somma di *Token Embedding* e *Positional Embedding* per fornire al modello sia l'identità del carattere che la sua coordinata spaziale nella sequenza.
* **Deep Stacking**: Utilizzo di una sequenza di blocchi Transformer (es. 8 blocchi) per incrementare la profondità della rappresentazione.
* **Output Head**: Proiezione lineare finale verso lo spazio del vocabolario per la generazione dei logit.
* **Inference Autoregressiva**: Generazione di nuovo testo partendo da un seed iniziale, campionando iterativamente dalla distribuzione di probabilità predetta.

PS: Ricorda dropout in HEAD nel prodotto qk e in multi head, `p = 0.2 `

PS: Module list per la multi head, poi torch.cat

PS: Ricorda la projection alla fine di tutto per riportare da emb_len a n_tokens

## Transoformers stuff



In [10]:
class Head(torch.nn.Module):
    def __init__(self, embedding_len, head_size):
        super().__init__()
        self.query = torch.nn.Linear(embedding_len, head_size, bias = False)
        self.key   = torch.nn.Linear(embedding_len, head_size, bias = False)
        self.value = torch.nn.Linear(embedding_len, head_size, bias = False)

        self.register_buffer("tril", torch.tril(torch.ones(context_len, context_len)))
        self.dropout = torch.nn.Dropout(p=0.2)
        self.sm = torch.nn.Softmax(dim=-1)
    
    def forward(self, x):
        # input x [batch_size, context_len, embedding_len]
        B, T, C = x.shape
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        d = k.shape[-1]

        qk = q @ k.transpose(-2, -1)         
        qk = qk / (d ** 0.5)
        
        # trick 
        qk = qk.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        qk = self.sm(qk)
         
        qk = self.dropout(qk)
        
        return qk @ v

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, head_size):
        super().__init__()
        self.heads = torch.nn.ModuleList([Head(embedding_len, head_size) for _ in range(num_heads)])
        self.dropout = torch.nn.Dropout(p=0.2)
    def forward(self,x): 
        out = torch.cat([head(x) for head in self.heads], dim = -1) 
        out = self.dropout(out)
        return out


class FeedForward(torch.nn.Module):
    # ff part of the transformer
    def __init__(self, embedding_len):
        super().__init__()
        self.l1 = torch.nn.Linear(embedding_len, 4 * embedding_len)
        self.l2 = torch.nn.Linear(4 * embedding_len, embedding_len)

        self.r1 = torch.nn.ReLU()
        self.r2 = torch.nn.ReLU()
        
    def forward(self,x):
        x = self.l1(x)
        x = self.r1(x)
        x = self.l2(x)
        x = self.r2(x)
        
        return x

class Block(torch.nn.Module):
    def __init__(self, embedding_len, num_heads):
        super().__init__()
        assert embedding_len % num_heads == 0, "For this example embedding len and num_heads should be multiple"
        self.ma = MultiHeadAttention(embedding_len, num_heads, embedding_len // num_heads)
        self.ln1 = torch.nn.LayerNorm(embedding_len)
        self.ff  = FeedForward(embedding_len) 
        self.ln2 = torch.nn.LayerNorm(embedding_len)

    def forward(self,x):
        x = self.ln1(x)
        x = x + self.ma(x)
        
        x = self.ln2(x)
        x = x + self.ff(x)

        return x


class PicoGPT(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, num_blocks, vocab_size, context_len):
        # embeddings and positional embeddings
        super().__init__()
        self.token_emb_table = torch.nn.Embedding(vocab_size,embedding_len) 
        self.pos_emb_table   = torch.nn.Embedding(context_len, embedding_len)
        self.blocks = torch.nn.Sequential(*[Block(embedding_len, num_heads) for _ in range(num_blocks)])
        self.ln = torch.nn.LayerNorm(embedding_len)
        self.projection = torch.nn.Linear(embedding_len, vocab_size)

        # use unpack operator to get the thing unpacked in the args

    
    def forward(self,data):
        B, T = data.shape
        data_idx = torch.arange(T).to(device)
        
        # retrieve the context len
        x = self.token_emb_table(data) + self.pos_emb_table(data_idx)
        x = self.blocks(x)

        return self.projection(self.ln(x))


        

In [11]:
num_heads = 6
embedding_len = 384
num_blocks = 8
num_iters = 1000

model = PicoGPT(embedding_len=embedding_len, num_heads=num_heads, num_blocks=num_blocks, vocab_size=vocab_size, context_len=context_len)
loss = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr = 3e-4)



In [13]:
model.to(device)
model.train()
every = 100
loss_vals = [0. in range(num_iters)]
for i in range(num_iters):
    # training loop
    optim.zero_grad()
    x,y = get_batch("train")
    logits = model(x)

    B,T,C = logits.shape

    loss_val = loss(logits.view(B*T,C), y.view(B*T))
    loss_vals.append(loss_val.item())
    loss_val.backward()
    optim.step()
    
  
    if i% every == 0: print(f"Loss @ iteration {i} : {loss_val}")

Loss @ iteration 0 : 4.25588321685791
Loss @ iteration 100 : 2.4753849506378174
Loss @ iteration 200 : 2.4109034538269043
Loss @ iteration 300 : 2.2173354625701904
Loss @ iteration 400 : 2.083483934402466
Loss @ iteration 500 : 1.9422169923782349
Loss @ iteration 600 : 1.864564061164856
Loss @ iteration 700 : 1.7643581628799438
Loss @ iteration 800 : 1.7204763889312744
Loss @ iteration 900 : 1.6740587949752808


In [32]:
#generation step

seq = "the cat"
seq_token = encode(seq).to(device).unsqueeze(0)
seq_token

tensor([[58, 46, 43,  1, 41, 39, 58]], device='cuda:0')

In [33]:
seq_token
sf = torch.nn.Softmax(dim = -1)
model.eval()
for i in range(len(seq_token) - 1, 200):
    logits = model(seq_token)
    next_token = torch.multinomial(sf(logits[:,-1,:]), 1).item()
    seq_token = torch.cat((seq_token, torch.tensor(next_token).reshape(1,1).to(device)), dim=1).to(device)
    #print(decode_id(seq_token[0].numpy()))


In [34]:
seq_token.shape

torch.Size([1, 207])

In [35]:

print(decode(seq_token[0].cpu().numpy()))

the catains of thy.
Boys, by lay we let flend Edward's a from grief.

RAJULIET:
No, dead nothing hown 'fickly?

JULIET:
A so, both Marius; I think will not a boldift:
Come, my shall prefuter persecy hand mot
